In [ ]:
# [Cell 1: Setup & Libraries]
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 11

CLASSES = {0: "Pothole", 1: "Crack", 2: "Manhole"}
print("✅ Libraries imported.")

In [ ]:
# [Cell 2: Multi-Model Thesis Comparison Table]
EXPERIMENTS = {
    "YOLO12-Nano": "results/acc1_yolo12n_results/results.csv",
    "YOLO12-Small": "results/acc2_yolo12s_results/results.csv",
}

summary = []
for model_name, path in EXPERIMENTS.items():
    if Path(path).exists():
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
        summary.append({
            "Model Variant": model_name,
            "Best Epoch": int(best_row["epoch"]),
            "Precision": round(best_row["metrics/precision(B)"], 4),
            "Recall": round(best_row["metrics/recall(B)"], 4),
            "mAP@50": round(best_row["metrics/mAP50(B)"], 4),
            "mAP@50-95": round(best_row["metrics/mAP50-95(B)"], 4)
        })

df_results = pd.DataFrame(summary)
display(df_results)

# Export LaTeX table code for your thesis document
print("\n--- LaTeX Table Output ---")
print(df_results.to_latex(index=False))

In [ ]:
# [Cell 3: Validation mAP Progression Plot]
plt.figure(figsize=(10, 5))
for model_name, path in EXPERIMENTS.items():
    if Path(path).exists():
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        plt.plot(df["epoch"], df["metrics/mAP50(B)"], label=f"{model_name} mAP@50", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Validation mAP@50")
plt.title("mAP@50 Progression Across Model Variants")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("results/thesis_map50_curves.png", bbox_inches="tight")
plt.show()